In [2]:
# Interfaz
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np
import scipy.signal as sig
from NNMF_GIF2 import NNMF_GIF
import Funciones as Funciones
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
import time
# =========================================
cmap = Funciones.parula_map
color_ACC = 'blue'
color_EGG = 'orange'
color_dEGG = 'g'
color_OVV = 'magenta'
color_speech = 'black'
legend_loc = 'upper right'
color1 = 'red'
color2 = 'blue'
color3 = 'k'
line_width = 1.5

SP1_ = []



tab_resultados = widgets.Output()
def ejecutar(b):
    #==============================================================================================================================================================
    fs_r = 8000
    to = 0
    tf = 50
    tiempos = np.arange(to, tf, 1000/fs_r)
    archivo1 = Path(fc1.selected)
    nombre_archivo1 = archivo1.name
    datos1 = np.loadtxt(archivo1, delimiter='\t', skiprows=1)
    señal_MBK1  = datos1[:, 0] - np.mean(datos1[:, 0])
    señal_ACC1  = -datos1[:, 1]
    señal_ACC1  = señal_ACC1 - np.mean(señal_ACC1)
    señal_EGG1  = datos1[:, 3]
    señal_dEGG1 = np.gradient(señal_EGG1)
    if max(señal_dEGG1) > 0:
        señal_dEGG1 = señal_dEGG1 / max(señal_dEGG1)

    # ==============================================================================================================================================================

    betas = np.arange(beta.value[0],beta.value[1],0.01)
    i=0

    for beta_ in betas:
        progress.value = i   # actualizamos el valor
        time.sleep(0.3)      # pausa para ver la animación
        SP1 = NNMF_GIF(speech=señal_MBK1,
                    acelerometro=señal_ACC1,
                    dEGG = señal_dEGG1,
                    tiempos=tiempos,
                    fs=fs_r,
                    norma=norma.value,
                    pre_iteraciones=pre_iteraciones.value,
                    iteraciones=iteraciones.value,
                    p_filtro=p_filtro.value,
                    alfa=alfa.value,
                    beta_speech=beta_,
                    beta_acc= beta_,
                    pre_enfasis_speech=pre_enfasis_speech.value,
                    pre_enfasis_acc=pre_enfasis_acc.value,
                    duracion=duracion_ventana.value,
                    guardar_proceso=guardar_proceso.value,
                    normalizar_espectrogramas=normalizar_espectrogramas.value
        )

        fig1,ax1 = plt.subplots(4,1,figsize=(12,9))
        ax1[0].set_title(nombre_archivo1.removesuffix(".txt")+" - β ="+f"{round(beta_,2)}")
        #========================= Espectrograma combinado
        im4 = ax1[0].imshow(SP1.espectrograma_plot_original,
                        cmap = cmap,
                        origin='lower',
                        aspect='auto',
                        extent=[SP1.to_espectrograma_original,SP1.tf_espectrograma_original,
                                SP1.fo_espectrograma,2*SP1.ff_espectrograma])
        yticks = [0, 2000, 4000, 6000, 8000]
        ax1[0].set_yticks(yticks)
        bbox = ax1[0].get_position()
        width = 0.015
        eps = 0.01
        cax = fig1.add_axes([bbox.x1 + eps, bbox.y0, width, bbox.height])
        fig1.colorbar(im4, cax=cax)
        ax1[0].set_yticklabels(['0', '2000', '4000', '2000', '4000'])
        ax1[0].set_ylabel('Frecuencia [Hz]')
        # ------------- dEGG   
        ax0_ = ax1[0].twinx()
        ax0_.plot(SP1.tiempos_espectrograma_original,SP1.dEGG_original,color='white',label=r"$dEGG$",linewidth=line_width);
        ax0_.legend(loc='upper right')
        ax0_.set_xlim(SP1.to_espectrograma_original,SP1.tf_espectrograma_original)
        ax0_.set_ylim([-2,2])
        ax0_.set_xticks(np.arange(SP1.to_espectrograma_original,SP1.tf_espectrograma_original,SP1.duracion_ventana))
        ax0_.set_yticks([])
        ax0_.set_xlabel('Tiempo [ms]')

        #========================= Espectrograma aproximado sincronizado
        im2 = ax1[1].imshow(SP1.espectrograma_aprox_plot,
                        cmap=cmap,
                        origin='lower',
                        aspect='auto',
                        extent=[SP1.to_espectrograma_sincronizado, SP1.tf_espectrograma_sincronizado,
                                SP1.fo_espectrograma, SP1.ff_espectrograma])
        ax1[1].set_ylabel('Frecuencia [Hz]')
        ax1[1].set_xlabel("Tiempo [ms]")
        bbox = ax1[1].get_position()
        width = 0.015
        eps = 0.01
        cax = fig1.add_axes([bbox.x1 + eps, bbox.y0, width, bbox.height])
        fig1.colorbar(im2, cax=cax)
        # ------------- dEGG
        ax2_ = ax1[1].twinx()
        ax2_.plot(SP1.tiempos_espectrograma_sincronizado,SP1.dEGG_sincronizado,color='white', label=r"$dEGG$", linewidth=line_width)
        ax2_.legend(loc=legend_loc)
        ax2_.set_xlim(SP1.to_espectrograma_sincronizado, SP1.tf_espectrograma_sincronizado)
        ax2_.set_ylim([-2, 2])
        ax2_.set_xticks(np.arange(SP1.to_espectrograma_sincronizado, SP1.tf_espectrograma_sincronizado, SP1.duracion_ventana))
        ax2_.set_yticks([])

        xbbox = 1.27
        ybbox = 0.5 
        #========================= Flujos speech
        ax1[2].plot(SP1.tiempos_gve,SP1.flujo1_speech,color=color1,label=r'$\hat{u}_{g1}(t)$ - speech')
        ax1_ = ax1[2].twinx()
        ax1_.plot(SP1.tiempos_gve,SP1.flujo2_speech,color=color2,label=r'$\hat{u}_{g2}(t)$ - speech')
        ax1[2].set_xlim([to,tf])
        ax1[2].set_xticks(np.arange(to,tf,2.5),minor=True)
        ax1[2].grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
        h1, l1 = ax1[2].get_legend_handles_labels()
        h2, l2 = ax1_.get_legend_handles_labels()
        ax1[2].legend(h1 + h2, l1 + l2, loc=legend_loc)
        ax1[2].set_xlabel('Tiempo [ms]')
        #========================= Flujos ACC
        ax1[3].plot(SP1.tiempos_gve,SP1.flujo1_acc,color=color1,label=r'$\hat{u}_{g1}(t)$ - acc')
        ax2_ = ax1[3].twinx()
        ax2_.plot(SP1.tiempos_gve,SP1.flujo2_acc,color=color2,label=r'$\hat{u}_{g2}(t)$ - acc')
        ax1[3].set_xlim([to,tf])
        ax1[3].set_xticks(np.arange(to,tf,2.5),minor=True)
        ax1[3].grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
        h1, l1 = ax1[3].get_legend_handles_labels()
        h2, l2 = ax2_.get_legend_handles_labels()
        ax1[3].legend(h1 + h2, l1 + l2, loc=legend_loc)
        ax1[3].set_xlabel('Tiempo [ms]')


        fig1.savefig("Animacion beta/A/"+str(i)+".pdf",bbox_inches = 'tight')
        plt.close();
        i+=1
#==================================================================
# FileChooser
#==================================================================

fc1 = FileChooser('Señales AC3E/Femenino')
fc1.layout = widgets.Layout(width='auto')


#==================================================================
# Widgets
#==================================================================

norma = widgets.Dropdown(
    options=['frobenius','itakura-saito'],
    value='frobenius',
    description='Norma:',
    layout=widgets.Layout(width='300px')
)
ventana = widgets.Dropdown(
    options=['boxcar','hanning'],
    value='boxcar',
    description='Ventana:',
    layout=widgets.Layout(width='300px')
)
duracion_ventana = widgets.BoundedFloatText(
    value=5.5,
    min=1.0,
    max=20.0,
    step=0.5,
    description='Duración ventana (ms):',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
desplazamiento = widgets.BoundedIntText(
    value=1,
    min=1,
    max=10,
    step=1,
    description='hop:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px'),
)
pre_iteraciones = widgets.BoundedIntText(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Pre-iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)

iteraciones = widgets.BoundedIntText(
    value=250,
    min=1,
    max=50000,
    step=1,
    description='Iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)

p_filtro = widgets.BoundedIntText(
    value=12,
    min=6,
    max=18,
    step=1,
    description='Orden del filtro del tracto:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)

alfa = widgets.BoundedFloatText(
    value=0.96,
    min=0.0,
    max=1.0,
    step=0.01,
    description='α:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px')
)

beta_speech = widgets.BoundedFloatText(
    value=0.20,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)

beta_acc = widgets.BoundedFloatText(
    value=0.55,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
beta = widgets.FloatRangeSlider(
    value=[0.20, 2.00],
    min=0.20,
    max=2.50,
    step=0.01,
    description='β:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
)


pre_enfasis_speech = widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)

pre_enfasis_acc = widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)

guardar_proceso = widgets.Checkbox(
    value=False,
    description='Guardar proceso'
)

normalizar_espectrogramas = widgets.Checkbox(
    value=True,
    description='Normalizar espectrogramas'
)

guardar = widgets.Checkbox(
    value=False,
    description='Guardar'
)

boton = widgets.Button(
    description="Ejecutar",
    button_style='success'
)

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(np.arange(beta.value[0],beta.value[1],0.01))-1,
    step=1,
    description='Cargando:',
    bar_style='',  # 'success', 'info', 'warning', 'danger'
    orientation='horizontal'
)
# Función que actualiza el máximo de la barra
def update_progress_max(change):
    start, end = change['new']
    progress.max = len(np.arange(start, end, 0.01)) - 1

# Observador: cada vez que cambia el slider, recalcula el max
beta.observe(update_progress_max, names='value')

b_ovv = widgets.BoundedIntText(
    value=70,
    min=10,
    max=200,
    step=1,
    description='Ancho de banda [Hz]:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)

#==================================================================
# Firulos estéticos
#==================================================================
titulo = widgets.HTML(
    value="<b>Archivo</b>"
)

linea_divisoria = widgets.HTML(
    value="<hr style='border:1px solid #ccc; margin:10px 0;'>"
)

subtitulo_espectrograma = widgets.HTML(
    value="<b>Espectrograma</b>"
)
subtitulo_speech= widgets.HTML(
    value="<b>Speech</b>"
)
subtitulo_acc= widgets.HTML(
    value="<b>ACC</b>"
)
subtitulo_nmf= widgets.HTML(
    value="<b>NNMF</b>"
)
subtitulo_gif= widgets.HTML(
    value="<b>GIF</b>"
)
subtitulo_ovv= widgets.HTML(
    value="<b>OVV</b>"
)
grid_speech = widgets.GridBox(
    children=[pre_enfasis_speech],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',   # ancho automático
        grid_gap='0px 5px',         # 0px entre filas, 5px entre columnas
        width='40%'
    )
)

grid_acc = widgets.GridBox(
    children=[pre_enfasis_acc],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 5px',
        width='40%'
    )
)

grid_espectrograma = widgets.GridBox(
    children=[ventana, duracion_ventana, desplazamiento,normalizar_espectrogramas],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',   # aquí puedes ajustar distinto si quieres más separación
        width='70%'
    )
)
grid_nmf = widgets.GridBox(
    children=[norma, pre_iteraciones, iteraciones],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',   # aquí puedes ajustar distinto si quieres más separación
        width='70%'
    )
)

grid_gif = widgets.GridBox(
    children=[p_filtro, alfa],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',   # aquí puedes ajustar distinto si quieres más separación
        width='40%'
    )
)

grid_ovv = widgets.GridBox(
    children=[b_ovv],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',   # aquí puedes ajustar distinto si quieres más separación
        width='40%'
    )
)
grid = widgets.GridBox(
    children=[fc1],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='row',
        grid_gap='10px 50px',   # aquí puedes ajustar distinto si quieres más separación
        width='60%'
    )
)
grid_boton = widgets.GridBox(
    children=[boton,progress,beta],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_gap='10px 50px',   # aquí puedes ajustar distinto si quieres más separación
        width='60%'
    )
)

# Disposición final
configuracion = widgets.VBox([
    titulo,
    grid,
    grid_boton,
    linea_divisoria,
    subtitulo_speech,
    grid_speech,
    linea_divisoria,
    subtitulo_acc,
    grid_acc,
    linea_divisoria,
    subtitulo_espectrograma,
    grid_espectrograma,
    linea_divisoria,
    subtitulo_nmf,
    grid_nmf,
    linea_divisoria,
    subtitulo_gif,
    grid_gif,
    linea_divisoria,
    subtitulo_ovv,
    grid_ovv
    #subtitulo_opciones,
    #checks,

    #boton
])

# Datos de ejemplo
x = np.linspace(0, 2*np.pi, 100)



tab_resultados = widgets.Output()


# --- Creamos el Tab ---





#==================================================================
# Outputs de plots
#==================================================================




#==================================================================
# Función ejecutar
#==================================================================
#==================================================================
# Tabs
#==================================================================

tab = widgets.Tab(children=[
    configuracion,
    tab_resultados],
)

tab.set_title(0,'Configuración')
tab.set_title(1,'Resultados')

#tab.layout = widgets.Layout(width='1200px')

boton.on_click(ejecutar)
display(tab)